# 04 — The improved model, built in steps

**Reads:** `data/raw/prices_yahoo.csv`.

**Writes:** `data/processed/calendar.csv` (rebalance and entry dates), `data/processed/panel.csv` (features and
target for every stock-day), one prediction file per model in `data/processed/pred_<model>.csv`,
`results/tables/tuning_grid.csv`, `results/tables/improved_feature_importance.csv`.

The class model is changed one step at a time, so the evaluation in notebook 05 can show which change mattered:

| Model | What changes from the row above |
|---|---|
| `original` | the class script as written (notebook 03) |
| `clean` | **features and target.** Same 18 months of training data, same hyperparameters, one fit. Price levels and volume are dropped, momentum points the right way, every feature is ranked across stocks each day, and the target is the stock's 20-day return *minus the average stock's* |
| `long` | **training window.** The same model trained on 2011–2023 instead of mid-2022–2023 |
| `improved` | **tuning and updating.** Hyperparameters chosen on 2020–2023 validation data, and the model refitted every month on everything known up to then |
| `linear` | a simple model for comparison: ordinary least squares on the same features, refitted every month |
| `momentum` | a rule with no model: rank stocks by their return over the past year, skipping the last month |

The test period, January 2024 onward, is not used for any choice in this notebook. The steps above were fixed
before the test results were computed, and each model was run once.

In [1]:
from pathlib import Path
import itertools, json
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy import stats
from sklearn.linear_model import LinearRegression

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
TAB = ROOT / "results" / "tables"

SEED = 6400
SPLIT = pd.Timestamp("2024-01-01")        # nothing from this date on is used to design or tune a model
HORIZON = 20                              # trading days in the target
EMBARGO = HORIZON + 1                     # a training row's target must end on or before the prediction day
SAMPLE_EVERY = 5                          # long-history models train on every fifth trading day

px = pd.read_csv(RAW / "prices_yahoo.csv", parse_dates=["date"])
stocks = px[~px["symbol"].isin(["SPY", "^IRX"])]
adj = stocks.pivot(index="date", columns="symbol", values="adjusted")
dollar_volume = (stocks.pivot(index="date", columns="symbol", values="close")
                 * stocks.pivot(index="date", columns="symbol", values="volume"))
spy = px[px["symbol"] == "SPY"].set_index("date")["adjusted"]
ret = adj.pct_change(fill_method=None)
mkt = spy.pct_change()
days = adj.index
pos = pd.Series(np.arange(len(days)), index=days)        # position of each trading day in the calendar
adj.shape

(4242, 96)

## Calendar

Same rule as the class script: decide on the first trading day of each month (the **rebalance date**), using
prices up to that day's close. The position is taken at the **next** day's close (the **entry date**) and held
to the next month's entry date. A period is used for evaluation only if it is complete.

In [2]:
test_days = days[days >= SPLIT]
rebalance = pd.DatetimeIndex(pd.Series(test_days).groupby(test_days.to_period("M")).min().values)
entry = pd.DatetimeIndex([days[pos[d] + 1] for d in rebalance if pos[d] + 1 < len(days)])
calendar = pd.DataFrame({"rebalance": rebalance[:len(entry)], "entry": entry})
calendar["exit"] = calendar["entry"].shift(-1)            # the next entry date; missing for the latest signal
calendar.to_csv(PROC / "calendar.csv", index=False)
print(len(calendar), "rebalance dates;", calendar["exit"].notna().sum(), "complete holding periods")
calendar.iloc[[0, 1, -2, -1]]

35 rebalance dates; 34 complete holding periods


,rebalance,entry,exit
0,2024-01-02,2024-01-03,2024-02-02
1,2024-02-01,2024-02-02,2024-03-04
33,2026-10-01,2026-10-02,2026-11-03
34,2026-11-02,2026-11-03,NaT


## Features

Twelve signals, each computable from prices up to the close of day t. They are the standard price-based
signals in the asset-pricing literature: momentum, short-term reversal, volatility, market beta, distance from
the one-year high, the largest daily gain, and unusual trading volume.

In [3]:
var_m = mkt.rolling(126).var()
beta_6m = ret.rolling(126).cov(mkt).div(var_m, axis=0)
resid_var = ret.rolling(126).var() - (beta_6m ** 2).mul(var_m, axis=0)

features = {
    "mom_12_1": adj.shift(21) / adj.shift(252) - 1,          # past year, skipping the last month
    "mom_6_1": adj.shift(21) / adj.shift(126) - 1,           # past six months, skipping the last month
    "mom_3m": adj / adj.shift(63) - 1,                       # past three months
    "rev_1m": adj / adj.shift(21) - 1,                       # past month (short-term reversal)
    "rev_5d": adj / adj.shift(5) - 1,                        # past week
    "vol_1m": ret.rolling(21).std(),                         # s.d. of daily returns, one month
    "vol_6m": ret.rolling(126).std(),                        # s.d. of daily returns, six months
    "beta_6m": beta_6m,                                      # sensitivity to SPY, six months
    "ivol_6m": np.sqrt(resid_var.clip(lower=0)),             # volatility not explained by SPY
    "high_52w": adj / adj.rolling(252).max(),                # price relative to its one-year high
    "max_1m": ret.rolling(21).max(),                         # largest daily gain in the past month
    "dvol_trend": dollar_volume.rolling(21).mean() / dollar_volume.rolling(252).mean(),   # recent vs usual trading
}
FEATURES = list(features)

# the target: return from tomorrow's close to the close 20 trading days later
fwd = adj.shift(-(HORIZON + 1)) / adj.shift(-1) - 1

panel = pd.concat({k: v.stack(future_stack=True) for k, v in features.items()}, axis=1)
panel["fwd_ret"] = fwd.stack(future_stack=True)
panel = panel.dropna(subset=FEATURES).reset_index()         # a stock enters once it has a year of history

# rank every feature across stocks within each day, scaled to (-0.5, 0.5)
by_day = panel.groupby("date")
for f in FEATURES:
    panel[f] = (by_day[f].rank() - 0.5) / by_day[f].transform("size") - 0.5
# relative target: the stock's forward return minus the average stock's on the same day
panel["target"] = panel["fwd_ret"] - by_day["fwd_ret"].transform("mean")
panel["pos"] = panel["date"].map(pos)
panel.round(6).to_csv(PROC / "panel.csv", index=False)
print(f"{len(panel):,} stock-days, {panel['date'].min().date()} to {panel['date'].max().date()}, "
      f"{panel.groupby('date')['symbol'].size().min()} to {panel.groupby('date')['symbol'].size().max()} stocks a day")

380,937 stock-days, 2011-01-03 to 2026-11-12, 92 to 96 stocks a day


## Training rows without look-ahead

A row dated t has a target that ends 21 trading days later. To predict on day d, a row may be used only if its
target is already known on d, so only rows with t at least 21 trading days before d. The class script skips
this step; its last training rows have targets that end in late January 2024.

In [4]:
def training_rows(predict_day, first_day=None, every=1):
    """Rows whose target is known by `predict_day`, optionally from `first_day`, keeping one day in `every`."""
    keep = panel["pos"] <= pos[predict_day] - EMBARGO
    if first_day is not None:
        keep &= panel["date"] >= first_day
    if every > 1:
        keep &= panel["pos"] % every == 0
    return panel[keep & panel["target"].notna()]


def daily_rank_ic(frame, score):
    """Average over days of the rank correlation between a score and the target."""
    return frame.assign(score=score).groupby("date").apply(
        lambda g: stats.spearmanr(g["score"], g["target"])[0], include_groups=False).mean()


ORIGINAL_PARAMS = dict(n_estimators=300, max_depth=5, learning_rate=0.05, gamma=0, min_child_weight=10,
                       subsample=1.0, colsample_bynode=1.0)
signal_rows = panel[panel["date"].isin(calendar["rebalance"])]        # the rows to be scored


def fit_xgb(rows, params):
    model = xgb.XGBRegressor(objective="reg:squarederror", n_jobs=4, random_state=SEED, **params)
    return model.fit(rows[FEATURES], rows["target"])


def save_predictions(name, scores):
    out = signal_rows[["date", "symbol"]].assign(pred=scores)
    out.to_csv(PROC / f"pred_{name}.csv", index=False)
    return out

## Step 1 — `clean`: new features and target, everything else as in the class script

Training window from 6 July 2022 (the class script's first training day) to the last day whose target is known
before the test period starts. Daily rows, the class hyperparameters, one fit.

In [5]:
first_test_day = calendar["rebalance"].iloc[0]
rows_clean = training_rows(first_test_day, first_day=pd.Timestamp("2022-07-06"))
model_clean = fit_xgb(rows_clean, ORIGINAL_PARAMS)
save_predictions("clean", model_clean.predict(signal_rows[FEATURES]))
print(f"clean: {len(rows_clean):,} training rows, {rows_clean['date'].min().date()} to {rows_clean['date'].max().date()}")

clean: 34,080 training rows, 2022-07-06 to 2023-11-30


## Step 2 — `long`: the same model on thirteen years

From the first day with a full year of history (January 2011). Twenty-day targets on neighbouring days overlap
almost completely, so the long-history models train on every fifth trading day.

In [6]:
rows_long = training_rows(first_test_day, every=SAMPLE_EVERY)
model_long = fit_xgb(rows_long, ORIGINAL_PARAMS)
save_predictions("long", model_long.predict(signal_rows[FEATURES]))
print(f"long: {len(rows_long):,} training rows, {rows_long['date'].min().date()} to {rows_long['date'].max().date()}")

long: 61,980 training rows, 2011-01-06 to 2023-11-29


## Step 3 — `improved`: hyperparameters tuned on pre-2024 validation data

Two validation folds, both before the test period: train to the end of 2019 and validate on 2020–2021; train to
the end of 2021 and validate on 2022–2023 (with the same 21-day gap). The score is the average daily rank
correlation between prediction and outcome, because the portfolio depends on the ordering of stocks and not on
the size of the predicted return. The grid leans towards smaller trees, slower learning and larger leaves than
the class settings: with a signal this weak, a flexible model fits noise.

In [7]:
FOLDS = [("2019-12-31", "2020-01-01", "2021-12-31"), ("2021-12-31", "2022-01-01", "2023-11-29")]
GRID = dict(max_depth=[2, 3, 5], learning_rate=[0.01, 0.05], min_child_weight=[20, 100])
TREE_COUNTS = [100, 300]
FIXED = dict(gamma=0, subsample=0.7, colsample_bytree=0.7)

sampled = panel[(panel["pos"] % SAMPLE_EVERY == 0) & panel["target"].notna()]
grid_rows = []
for depth, rate, leaf in itertools.product(*GRID.values()):
    for train_end, val_start, val_end in FOLDS:
        first_val_pos = pos[days[days >= val_start][0]]
        tr = sampled[sampled["pos"] <= first_val_pos - EMBARGO]
        va = sampled[(sampled["date"] >= val_start) & (sampled["date"] <= val_end)]
        model = fit_xgb(tr, dict(n_estimators=max(TREE_COUNTS), max_depth=depth, learning_rate=rate,
                                 min_child_weight=leaf, **FIXED))
        for trees in TREE_COUNTS:                      # score the first 100 trees and all 300 from one fit
            score = model.predict(va[FEATURES], iteration_range=(0, trees))
            grid_rows.append(dict(max_depth=depth, learning_rate=rate, min_child_weight=leaf, n_estimators=trees,
                                  fold=val_start[:4] + "-" + val_end[:4], rank_ic=daily_rank_ic(va, score)))

grid = (pd.DataFrame(grid_rows)
        .pivot_table(index=["max_depth", "learning_rate", "min_child_weight", "n_estimators"],
                     columns="fold", values="rank_ic"))
grid["mean_rank_ic"] = grid.mean(axis=1)
grid = grid.sort_values("mean_rank_ic", ascending=False)
grid.round(4).to_csv(TAB / "tuning_grid.csv")

# the class settings on the same folds, for reference
ref = []
for train_end, val_start, val_end in FOLDS:
    first_val_pos = pos[days[days >= val_start][0]]
    tr = sampled[sampled["pos"] <= first_val_pos - EMBARGO]
    va = sampled[(sampled["date"] >= val_start) & (sampled["date"] <= val_end)]
    ref.append(daily_rank_ic(va, fit_xgb(tr, ORIGINAL_PARAMS).predict(va[FEATURES])))
print("class hyperparameters on the validation folds:", np.round(ref, 4), "mean", round(float(np.mean(ref)), 4))

best = dict(zip(grid.index.names, grid.index[0]))
BEST_PARAMS = dict(max_depth=int(best["max_depth"]), learning_rate=float(best["learning_rate"]),
                   min_child_weight=int(best["min_child_weight"]), n_estimators=int(best["n_estimators"]), **FIXED)
json.dump({"chosen": BEST_PARAMS, "validation_rank_ic": float(grid["mean_rank_ic"].iloc[0]),
           "class_params_validation_rank_ic": float(np.mean(ref))}, open(TAB / "tuning_choice.json", "w"), indent=2)
print("chosen:", BEST_PARAMS)
grid.head(8).round(4)

class hyperparameters on the validation folds: [0.0224 0.0325] mean 0.0275
chosen: {'max_depth': 2, 'learning_rate': 0.01, 'min_child_weight': 20, 'n_estimators': 100, 'gamma': 0, 'subsample': 0.7, 'colsample_bytree': 0.7}


fold                                                   2020-2021  2022-2023  \
max_depth learning_rate min_child_weight n_estimators                         
2         0.01          20               100              0.0485     0.0440   
                        100              100              0.0491     0.0433   
3         0.01          100              100              0.0436     0.0486   
2         0.01          100              300              0.0458     0.0463   
5         0.01          20               100              0.0520     0.0400   
2         0.01          20               300              0.0462     0.0446   
5         0.01          100              100              0.0442     0.0464   
3         0.01          20               100              0.0444     0.0420   

fold                                                   mean_rank_ic  
max_depth learning_rate min_child_weight n_estimators                
2         0.01          20               100                 0.0463  
                        100              100                 0.0462  
3         0.01          100              100                 0.0461  
2         0.01          100              300                 0.0461  
5         0.01          20               100                 0.0460  
2         0.01          20               300                 0.0454  
5         0.01          100              100                 0.0453  
3         0.01          20               100                 0.0432

## Step 3, continued — refit every month (walk-forward)

On each rebalance date the model is trained again on every row whose target is known by then, and scores that
day's stocks. The model that picks the November 2026 portfolio has seen data to early October 2026; the class
model stops learning in December 2023. The linear model is refitted on the same rows.

In [8]:
pred_improved = pd.Series(index=signal_rows.index, dtype=float)
pred_linear = pd.Series(index=signal_rows.index, dtype=float)
for d in calendar["rebalance"]:
    rows = training_rows(d, every=SAMPLE_EVERY)
    today = signal_rows["date"] == d
    model = fit_xgb(rows, BEST_PARAMS)
    pred_improved[today] = model.predict(signal_rows.loc[today, FEATURES])
    pred_linear[today] = LinearRegression().fit(rows[FEATURES], rows["target"]).predict(signal_rows.loc[today, FEATURES])

save_predictions("improved", pred_improved)
save_predictions("linear", pred_linear)
save_predictions("momentum", signal_rows["mom_12_1"])

importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
importance.rename("gain_share").round(3).to_csv(TAB / "improved_feature_importance.csv", index_label="feature")
print(f"last fit: {len(rows):,} rows to {rows['date'].max().date()}")
importance.round(3)

last fit: 75,612 rows to 2026-09-30


vol_6m        0.189
ivol_6m       0.137
mom_6_1       0.095
mom_12_1      0.086
max_1m        0.076
high_52w      0.073
dvol_trend    0.073
rev_1m        0.072
beta_6m       0.067
vol_1m        0.057
mom_3m        0.038
rev_5d        0.037
dtype: float32